# New Structure Chatbot-Style Answer Tester

Type a question, run it through the normal classification/source preparation step, then answer it using the new `planning/new_structure` slot planner.

This notebook does **not** replace the chatbot runtime. It is only for checking whether the new structure can produce good chatbot-style answers.

In [80]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

import importlib

import pandas as pd
from IPython.display import display, JSON, Markdown

from historyAssistant.app.runtime import load_runtime
from historyAssistant.app.chatbot_service import classify_and_prepare_source
from historyAssistant.app.narration import narrate
from historyAssistant.config import load_source_configs
from historyAssistant.execution.chart_renderer import render_chart
from historyAssistant.execution.executor import open_es_connection, execute_canonical_plan
from historyAssistant.llm import load_source_examples
from historyAssistant.planning import normalize_question
import historyAssistant.planning.new_structure.intent_slots as _intent_slots
import historyAssistant.planning.new_structure.neutral_intent as _neutral_intent
import historyAssistant.planning.new_structure.value_filter_inference as _value_filter_inference
import historyAssistant.planning.new_structure.llm_slot_extractor as _llm_slot_extractor
import historyAssistant.planning.new_structure.intent_extractor as _new_intent_extractor
import historyAssistant.planning.new_structure.slot_planner as _new_slot_planner

# Helpful while iterating on the new structure: rerunning this setup cell refreshes local code changes.
importlib.reload(_intent_slots)
importlib.reload(_neutral_intent)
importlib.reload(_value_filter_inference)
importlib.reload(_llm_slot_extractor)
importlib.reload(_new_intent_extractor)
importlib.reload(_new_slot_planner)
try_build_slot_plan = _new_slot_planner.try_build_slot_plan

print('Project root:', ROOT)

Project root: c:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant


## Chatbot-Style Input

Edit `QUESTION`, then run the cells below. The notebook will classify the question, build a new-structure plan, execute it, and show a chatbot-style answer.

In [75]:
QUESTION = "calls in March 2024"

# Simple questions still use deterministic slots. Complex or invalid cases can use LLM-assisted intent confirmation.
USE_LLM_SLOTS = True

# Keep this True when you want to judge answer quality.
RUN_ANSWER = True

# Set True if you want LLM narration; False makes the wording deterministic and faster.
USE_LLM_NARRATION = True

# Optional: force one source when you want to test answer quality for a domain.
# Use None, 'calls', 'locations', or 'workflows'.
SOURCE_OVERRIDE = None

print('Question:', QUESTION)

Question: calls in March 2024


In [76]:
bundle, client, examples = load_runtime()
debug_payload = {"question": QUESTION}

source_context = classify_and_prepare_source(
    clean_message=QUESTION,
    bundle=bundle,
    client=client,
    examples=examples,
    debug_payload=debug_payload,
)

if SOURCE_OVERRIDE:
    if SOURCE_OVERRIDE not in {'calls', 'locations', 'workflows'}:
        raise ValueError("SOURCE_OVERRIDE must be None, 'calls', 'locations', or 'workflows'.")
    override_bundle = load_source_configs(SOURCE_OVERRIDE)
    override_source_cfg = override_bundle.app_config.get('data_sources', {}).get(SOURCE_OVERRIDE, {})
    override_normalization = normalize_question(QUESTION, override_bundle.vocabulary)

    source_context.source_id = SOURCE_OVERRIDE
    source_context.source_bundle = override_bundle
    source_context.source_examples = load_source_examples(SOURCE_OVERRIDE) or examples
    source_context.canonical_table = override_source_cfg.get('canonical_table')
    source_context.normalized_message = override_normalization.normalized_question
    source_context.detected_entities = override_normalization.detected_entities
    debug_payload['source_override'] = {
        'source_id': SOURCE_OVERRIDE,
        'canonical_table': source_context.canonical_table,
        'normalized_message': source_context.normalized_message,
        'detected_entities': source_context.detected_entities,
    }

classification_view = {
    "source_override": SOURCE_OVERRIDE,
    "source_id": source_context.source_id,
    "canonical_table": source_context.canonical_table,
    "scenario": source_context.scenario,
    "request_type": source_context.request_type,
    "normalized_message": source_context.normalized_message,
    "detected_entities": source_context.detected_entities,
    "source_confidence": getattr(source_context.classification, "source_confidence", None),
    "classification_confidence": (source_context.classification.debug_updates.get("scenario_routing", {}) or {}).get("confidence"),
    "routing_path": (source_context.classification.debug_updates.get("scenario_routing", {}) or {}).get("raw_response"),
    "classification_timing": source_context.classification.debug_updates.get("classification_timing"),
}

display(Markdown('### Classification Output'))
display(JSON(classification_view, expanded=True))

### Classification Output

<IPython.core.display.JSON object>

In [77]:
slot_result = try_build_slot_plan(
    question=QUESTION,
    normalized_question=source_context.normalized_message,
    source_id=source_context.source_id,
    target_table=source_context.canonical_table,
    detected_entities=source_context.detected_entities,
    request_type=source_context.request_type,
    chart_suppressed=source_context.chart_suppressed,
    preferred_chart_type=source_context.preferred_chart_type,
    bundle=source_context.source_bundle,
    scenario=source_context.scenario,
    client=client,
    use_llm_slots=USE_LLM_SLOTS,
)

display(Markdown('### New Structure Result'))
display(JSON(slot_result.to_debug_dict(), expanded=False))

### New Structure Result

<IPython.core.display.JSON object>

## Compact View

In [78]:
plan = slot_result.canonical_plan or {}
validation = slot_result.validation or {}
source_resolution = slot_result.debug.get('source_intent_resolution', {})
llm_extraction = slot_result.debug.get('llm_slot_extraction')
llm_validation = slot_result.debug.get('llm_validation')
slot_extraction = slot_result.debug.get('slot_extraction', {})

compact = {
    "used": slot_result.used,
    "reason": slot_result.reason,
    "valid": validation.get("valid"),
    "source_id": source_resolution.get("source_id"),
    "source_changed": source_resolution.get("changed_source"),
    "deterministic_slot_reason": slot_extraction.get("reason"),
    "llm_slot_attempted": llm_extraction is not None,
    "llm_slot_used": bool(llm_extraction and llm_extraction.get("used") and slot_result.reason == (llm_extraction or {}).get("reason")),
    "llm_slot_reason": (llm_extraction or {}).get("reason"),
    "llm_validation_valid": (llm_validation or {}).get("valid"),
    "request_type": plan.get("request_type"),
    "metric": plan.get("metric"),
    "aggregation": plan.get("aggregation"),
    "group_by": plan.get("group_by"),
    "filters": plan.get("filters"),
    "date_rule": plan.get("date_rule"),
    "date_range": plan.get("date_range"),
    "compare_to": plan.get("compare_to"),
    "bucket_interval": plan.get("bucket_interval"),
    "bucket_summary": plan.get("bucket_summary"),
    "query_shape": plan.get("query_shape"),
    "entity_intersection": plan.get("entity_intersection"),
    "chart_spec": plan.get("chart_spec"),
    "errors": validation.get("errors"),
    "blocking_errors": validation.get("blocking_errors"),
    "repairs": validation.get("repairs"),
}

display(JSON(compact, expanded=True))

<IPython.core.display.JSON object>

## Chatbot-Style Answer

This executes the new-structure plan and uses the normal narration layer to produce an answer. Use this section to judge answer quality.

In [79]:
if not RUN_ANSWER:
    print('Set RUN_ANSWER = True to produce a chatbot-style answer.')
elif not slot_result.used:
    display(Markdown('### Answer'))
    display(Markdown('The new structure did not produce a plan for this question.'))
    display(JSON(slot_result.to_debug_dict(), expanded=True))
elif not slot_result.validation or not slot_result.validation.get('valid'):
    display(Markdown('### Answer'))
    display(Markdown('The new structure produced a plan, but validation rejected it.'))
    display(JSON(slot_result.to_debug_dict(), expanded=True))
else:
    source_resolution = slot_result.debug.get('source_intent_resolution', {})
    runtime_source_id = source_resolution.get('source_id') or source_context.source_id
    runtime_bundle = load_source_configs(runtime_source_id)
    es = open_es_connection(config_bundle=runtime_bundle)
    runtime_plan = slot_result.validation.get('canonical_plan') or slot_result.canonical_plan
    execution_result = execute_canonical_plan(runtime_plan, es, config_bundle=runtime_bundle)
    execution_result['plan'] = runtime_plan

    narration_debug = {}
    answer_text = narrate(
        QUESTION,
        execution_result,
        client,
        force_deterministic=not USE_LLM_NARRATION,
        debug_sink=narration_debug,
    )

    display(Markdown('### Answer'))
    display(Markdown(answer_text))

    dataframe = execution_result.get('dataframe')
    if dataframe is not None:
        display(Markdown('### Table'))
        display(dataframe)

    fig = render_chart(execution_result.get('chart_payload'))
    if fig:
        display(Markdown('### Chart'))
        fig.show()

    display(Markdown('### Answer Debug'))
    display(JSON({
        'narration_debug': narration_debug,
        'runtime_source_id': runtime_source_id,
        'target_tables': runtime_plan.get('target_tables'),
        'metric': runtime_plan.get('metric'),
        'result_type': execution_result.get('result_type'),
        'record_count': execution_result.get('total_count') or len(execution_result.get('records') or []),
        'scalar_answer': execution_result.get('scalar_answer'),
        'has_chart': bool(execution_result.get('chart_payload')),
        'error': execution_result.get('error'),
        'warnings': execution_result.get('warnings'),
    }, expanded=False))

c:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\src\historyAssistant\execution\es_builder.py:361: ElasticsearchWarning:

Elasticsearch built-in security features are not enabled. Without authentication, your cluster could be accessible to anyone. See https://www.elastic.co/guide/en/elasticsearch/reference/7.17/security-minimal-setup.html to enable security.

c:\Users\S_YZH\OneDrive - Televic Group NV\Desktop\HistoryAssistant\src\historyAssistant\execution\executor.py:286: ElasticsearchWarning:

Elasticsearch built-in security features are not enabled. Without authentication, your cluster could be accessible to anyone. See https://www.elastic.co/guide/en/elasticsearch/reference/7.17/security-minimal-setup.html to enable security.



### Answer

There were **5,425 calls in March 2024**.

### Table

,metric_value
0,5425


### Answer Debug

<IPython.core.display.JSON object>